# MX: evaluatie vanuit uitsluitend de opgeslagen E2M3-bron

**Geen training, geen herkalibratie, geen nieuwe checkpointselectie.**

Beide modi worden in een nieuw proces opgebouwd uit dezelfde hoge bron. De lage modus gebruikt `codes >> 2` met ongewijzigde schalen en de eigen lage BatchNorm-bank. Alleen de zes MX-mastertensors zijn afwezig; de uitzonderingslagen en BN-staat blijven FP32. Dit notebook leest uitsluitend de nieuwe evaluatierapporten.

In [ ]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

RUN_DIR = Path(".")  # RUN_DIRECTORY
report = json.loads((RUN_DIR / "run_report.json").read_text(encoding="utf-8"))
assert report["status"] == "PASS", "Controleer eerst het fout- of onderbrekingsrapport."
assert report["command"] == "evaluate", "Open de notebook uit de _evaluate-map, niet uit _check."
metrics = pd.read_csv(RUN_DIR / "metrics.csv")
compare = pd.read_csv(RUN_DIR / "historical_reproduction.csv")
display(Markdown("**Reproductiestatus:** " + report["reproduction_status"]))
if report.get("synthetic_fixture_only", False):
    display(Markdown("**SYNTHETISCHE SOFTWARETEST: onderstaande getallen zijn geen SVHN-resultaten.**"))

## 1. Scores: training tegenover herladen bron

De hoofdcontrole is epoch 100. De al eerder geselecteerde beste epoch wordt apart getoond. Er wordt geen nieuw gunstiger checkpoint gekozen. Een overeenkomst in accuracy is **geen bewijs van identieke logits** met de oude training: oude individuele logits zijn niet beschikbaar.

In [ ]:
cols = ["role", "epoch", "weight_bits", "training_accuracy_percent", "export_accuracy_percent", "difference_correct1", "ce_difference", "reproduction"]
display(compare[cols].round(6))

labels = [f"{r.role} | epoch {r.epoch} | E2M{3 if r.weight_bits == 6 else 1}" for r in compare.itertuples()]
x = np.arange(len(compare))
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.barh(x - 0.18, compare.training_accuracy_percent, height=0.36, label="Gerapporteerd na training")
ax.barh(x + 0.18, compare.export_accuracy_percent, height=0.36, label="Nieuw: alleen hoge bron")
ax.set_yticks(x, labels); ax.invert_yaxis()
ax.set_xlim(0, 100); ax.set_xlabel("Testaccuracy (%)")
ax.set_title("Reproduceert de hoge bron beide uitvoeringsmodi?")
ax.legend(loc="lower right")
fig.tight_layout(); plt.show()

## 2. Technische bron- en integriteitscontroles

De worker kan via de gecontroleerde model-loader alleen zijn staged hoge bestand laden. Encoders, gewichtsvoorbewerking, optimizerconstructie en backward zijn geblokkeerd. Dit zijn softwarecontroles, geen OS-sandbox. Codes en schalen worden gecontroleerd; het model en de BN-toestanden moeten tijdens inferentie onveranderd blijven.

In [ ]:
checks = []
for role, w in report["workers"].items():
    checks.append({"rol": role, "worker_status": w["status"], "modelbestanden_geladen": w["model_file_count"],
        "hoge_bron_enkel": w["both_modes_from_same_high_payload"], "masters_afwezig": w["six_fp32_masters_absent"],
        "BN_en_model_ongewijzigd": w["all_states_unchanged"], "geen_encoder": w["no_encoder_or_G_W_called"],
        "nieuwe_proces_id": w["pid"]})
display(pd.DataFrame(checks))
display(Markdown("**Oorspronkelijke invoer ongewijzigd:** " + str(report["input_files_unchanged"])))

## 3. Nieuwe voorspellingen per beeld

De NPZ-bestanden bevatten labels, indices, logits, voorspellingen en verlies per beeld voor deze nieuwe evaluatie. De vergelijking E2M1 tegenover E2M3 is gepaard; het zijn geen onafhankelijke modellen of seeds.

In [ ]:
paired = pd.read_csv(RUN_DIR / "paired_metrics.csv")
display(paired)
for role, w in report["workers"].items():
    suffix = "best" if role == "best_E2M3_test" else "last"
    path = RUN_DIR / ("worker_" + suffix) / "per_image_outputs.npz"
    assert hashlib.sha256(path.read_bytes()).hexdigest() == w["per_image_sha256"]
    with np.load(path, allow_pickle=False) as data:
        y = data["labels"]
        assert np.array_equal(data["indices"], np.arange(len(y)))
        for bit in (4, 6):
            p = data[f"E2M{3 if bit == 6 else 1}__predictions"]
            row = metrics[(metrics.role == role) & (metrics.weight_bits == bit)].iloc[0]
            assert int((p == y).sum()) == int(row.correct1)
print("Nieuwe individuele voorspellingen en bestandshashes gecontroleerd.")

## 4. Waarschuwingen en interpretatie

`PASS` betekent dat de technische uitvoering is voltooid. `MATCH_COUNTS_CE_CLOSE` vereist gelijke top-1/top-5-aantallen en CE binnen abs. 1e-6 / rel. 1e-5. Deze CE-tolerantie betreft alleen de diagnostische vergelijking met de oude verliesaggregatie. `REVIEW_DIFFERENCE` wordt niet verborgen of opgelost door gewichten, BatchNorm of het testprotocol aan te passen.

Deze test meet de export-naar-inferentieroute; hij levert geen extra trainingsseed, native low-bit snelheidsmeting of bewijs van continual learning.

In [ ]:
display(pd.DataFrame(report.get("warnings", [])))
display(Markdown(f"**Gemeten sessieduur:** {report["session_wall_seconds"]:.2f} s (opstarten, twee processen en vier evaluatiepaden samen; geen hardwarebenchmark)."))
if report["reproduction_status"] == "REVIEW_DIFFERENCE":
    display(Markdown("**Nader bekijken:** deel de feedbackbundel; verander niets aan de oorspronkelijke modellen."))
else:
    display(Markdown("Beide bronmodi reproduceren de correcte aantallen en de gerapporteerde loss binnen de vastgelegde CE-tolerantie. Historische logits zijn niet opnieuw vergeleken."))